# Build Coupled Series with Multivariatizer

`Multivariatizer` wraps a univariate generator and introduces cross-series mixing and lead-lag relationships.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from synforecast import Multivariatizer
from synforecast.generators import TSIGenerator

In [ ]:
base = TSIGenerator(
    min_length=256,
    max_length=256,
    freq="h",
    engine="polars",
    seed=7,
)
coupled = Multivariatizer(base=base, seed=42)
df = coupled.generate(n_series=4)
df.head()

The sampled recipe records the transformations, making each generated panel inspectable and reproducible from its seed.

In [ ]:
coupled.last_recipe

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for series_id in df["unique_id"].unique().sort().to_list():
    series = df.filter(df["unique_id"] == series_id).sort("ds")
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=series_id)
ax.set(title="Coupled synthetic series", xlabel="Time", ylabel="y")
ax.legend()
plt.show()

In [ ]:
wide = df.pivot(on="unique_id", index="ds", values="y").sort("ds")
np.corrcoef(wide.drop("ds").to_numpy().T)